# California Housing Prices Prediction with Machine Learning

### Data Preprocessing

In this notebook we prepare the dataset for machine learning models.

Raw data often contains issues such as missing values, categorical variables, or features that require transformation before they can be used by most machine learning algorithms.

The goal of this stage is to transform the dataset into a clean numerical representation suitable for model training.

The preprocessing steps include:

- Splitting the dataset into training and test sets
- Separating the target variable from the predictors
- Handling missing values in numerical features
- Encoding categorical variables
- Combining all processed features into the final training dataset

In [1]:
#Importing the libraries required for data preporcessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
#Loading the dataset

df = pd.read_csv("housing.csv")

### 1. Train–Test Split

Before performing any preprocessing steps, the dataset is divided into a **training set** and a **test set**:

* The **training set** will be used to fit the preprocessing pipeline and train the machine learning models.
* The **test set** will be reserved for evaluating model performance on unseen data.

Maintaining a separate test set helps provide an unbiased estimate of how well the model generalizes.

In [3]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

### 2. Separating Features and Target Variable from the Train Set

The target variable in this project is **`median_house_value`**, which represents the median house price in each district.

Since this is the variable the model will learn to predict, it must be separated from the input features.

The dataset is therefore divided into:

- **Predictor variables (features)** – used as inputs to the model
- **Target variable** – the value to be predicted

In [4]:
#Predictor variables  -> Change name to x_train = housing_predictors
x_train = train_set.drop("median_house_value", axis=1)

#Target variable --> Change name to y_train = housing_objective
y_train = train_set["median_house_value"]

### 3. Identifying Numerical and Categorical Features

The dataset contains both numerical and categorical variables.

Most features are numerical, describing characteristics of housing districts such as population, median income, or geographic coordinates. However, the feature **`ocean_proximity`** is categorical and represents the district's location relative to the ocean.

Since numerical and categorical variables require different preprocessing techniques, we separate them before building the preprocessing pipeline.

In [5]:
# Numerical features (housing_num before)
x_train_num = x_train.drop("ocean_proximity", axis=1).columns

#Categorical features (housing_cat before)
x_train_cat = ["ocean_proximity"]


### 4. Building the Preprocessing Pipeline

To organize the preprocessing workflow, we use **scikit-learn pipelines**.

Pipelines allow multiple preprocessing steps to be combined into a single object, making the workflow cleaner and easier to maintain. They also help prevent **data leakage**, since transformations are fitted only on the training data.

In this project, the preprocessing pipeline performs the following operations:

**Numerical features**
- Missing values in `total_bedrooms` are imputed using the **median**, which is more robust to skewed distributions than the mean.

**Categorical features**
- The variable `ocean_proximity` is transformed using **One-Hot Encoding**, converting each category into a binary feature.

These transformations are applied using a **ColumnTransformer**, which allows different preprocessing pipelines to be applied to different subsets of features.

In [6]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [7]:
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [8]:
preprocessing = ColumnTransformer([
    ("num", num_pipeline, x_train_num),
    ("cat", cat_pipeline, x_train_cat)
])

#### Fitting the Pipeline on the Training Data

The preprocessing pipeline is fitted using only the training data.

During this step, the pipeline learns the necessary transformation parameters, such as:

- the median value used for imputing missing data
- the set of categories present in the categorical feature

Fitting the pipeline exclusively on the training set prevents information from the test set from leaking into the training process.

In [9]:
#Change name to x_train_prepared
x_train_prepared = preprocessing.fit_transform(x_train)

#### Transforming the Test Set

After fitting the preprocessing pipeline on the training data, the same transformations must be applied to the test set.

In this step, the pipeline does **not learn new parameters**. Instead, it applies the transformations previously fitted on the training data:

* the median used for imputing missing values
* the categories detected for the One-Hot Encoding.

This approach ensures that the test data undergoes exactly the same preprocessing steps as the training data while preventing data leakage.

X_test = test_set.drop("median_house_value", axis=1)
y_test = test_set["median_house_value"]

X_train_prepared = preprocessing.fit_transform(X_train)
X_test_prepared = preprocessing.transform(X_test)

In [10]:
#Predictor variables in the testing data.
x_test = test_set.drop("median_house_value", axis=1)

#Target variable in the testing data.
y_test = test_set["median_house_value"]

In [11]:
#Pipeline transformation of the predictor variables in the testing data.

x_test_prepared = preprocessing.transform(x_test)

Checking the number of columns in both the training and the test data is the same.

In [12]:
x_train_prepared.shape

(16512, 13)

In [13]:
x_test_prepared.shape

(4128, 13)

#### Saving the Processed Data

To keep the workflow modular and reproducible, the processed training and test datasets are saved to disk.

This allows the next notebook to load the prepared data directly without repeating the preprocessing steps.

In [15]:
np.save("data/x_train_prepared.npy", x_train_prepared)
np.save("data/y_train.npy", y_train)

np.save("data/X_test_prepared.npy", x_test_prepared)
np.save("data/y_test.npy", y_test)

FileNotFoundError: [Errno 2] No such file or directory: 'data/x_train_prepared.npy'